In [1]:
import pandas as pd

# Loading Transaction_file

In [2]:
df_transaction = pd.read_csv(r"D:\Dubai\Dubai_Updated_data\Raw_Data\Transactions.csv")
df_transaction.head()

,transaction_id,procedure_id,trans_group_id,trans_group_ar,trans_group_en,procedure_name_ar,procedure_name_en,instance_date,property_type_id,property_type_ar,...,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
0,3-9-2006-163,9,3,هبات,Gifts,هبه,Grant,16-10-2006,4,فيلا,...,NaN,0,3162.42,12000000.0,3794.56,NaN,NaN,3.0,1.0,0.0
1,3-9-2019-2944,9,3,هبات,Gifts,هبه,Grant,13-11-2019,1,أرض,...,NaN,0,209.09,916659.0,4384.04,NaN,NaN,2.0,4.0,0.0
2,2-13-1999-347,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,22-03-1999,1,أرض,...,NaN,0,1062.72,1200000.0,1129.18,NaN,NaN,1.0,1.0,0.0
3,2-13-2001-547,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,23-07-2001,2,مبنى,...,NaN,0,1393.55,3500000.0,2511.57,NaN,NaN,5.0,1.0,0.0
4,2-13-2020-9477,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,30-11-2020,2,مبنى,...,NaN,0,278.71,2500000.0,8969.90,NaN,NaN,1.0,1.0,0.0


In [3]:
df_transaction.shape

(1661875, 46)

# Driving Year and Quarter Column from instance_date column

In [4]:
import pandas as pd

# Step 1: Convert to datetime (invalid → NaT)
df_transaction['instance_date'] = pd.to_datetime(
    df_transaction['instance_date'],
    format='%d-%m-%Y',
    errors='coerce'
)

# Step 2: Count invalid dates
invalid_count = df_transaction['instance_date'].isna().sum()
print(f"Number of invalid dates found: {invalid_count}")

# Step 3: Show problematic original values (important fix)
if invalid_count > 0:
    print("\nProblematic raw values:")
    print(df_transaction.loc[df_transaction['instance_date'].isna(), 'instance_date'].head())

# Step 4: Extract year & quarter using nullable integer (fix for float issue)
df_transaction['year'] = df_transaction['instance_date'].dt.year.astype('Int64')
df_transaction['quarter_num'] = df_transaction['instance_date'].dt.quarter.astype('Int64')

# Step 5: Create clean quarter column (avoid float like Q4.0-2006.0)
df_transaction['quarter'] = (
    'Q' + df_transaction['quarter_num'].astype(str) + '-' + df_transaction['year'].astype(str)
)

# Optional: Replace invalid rows with None instead of "Q<NA>-<NA>"
df_transaction.loc[df_transaction['instance_date'].isna(), 'quarter'] = None

# Step 6: Final check
print(df_transaction[['instance_date', 'year', 'quarter']].head())

Number of invalid dates found: 4

Problematic raw values:
25924    NaT
387853   NaT
388014   NaT
879662   NaT
Name: instance_date, dtype: datetime64[ns]
  instance_date  year  quarter
0    2006-10-16  2006  Q4-2006
1    2019-11-13  2019  Q4-2019
2    1999-03-22  1999  Q1-1999
3    2001-07-23  2001  Q3-2001
4    2020-11-30  2020  Q4-2020


# Driving property_type column from property_sub_type_en column

In [5]:
flat_types = {
    "Flat", "Villa", "Stacked Townhouses", "Unit"
}

shop_types = {
    "Shop", "Store", "Show Rooms"
}

office_types = {
    "Office", "Clinic", "Workshop", "Warehouse"
}

others = {
    "Hotel Apartment",
    "Hotel Rooms",
    "Gymnasium",
    "Sized Partition",
    "Hotel",
    "Building"
}
def categorize_property(sub_type):
    if sub_type in flat_types:
        return "Flat"
    elif sub_type in shop_types:
        return "Shop"
    elif sub_type in office_types:
        return "Office"
    else:
        return "Others"

    
df_transaction['property_type'] = df_transaction['property_sub_type_en'].apply(categorize_property)

In [9]:
df_transaction.sample(10)

,transaction_id,procedure_id,trans_group_id,trans_group_ar,trans_group_en,procedure_name_ar,procedure_name_en,instance_date,property_type_id,property_type_ar,...,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3,quarter,year,property_type
494,1-102-2025-57872,102,1,مبايعات,Sales,بيع - تسجيل مبدئى,Sell - Pre registration,2025-07-03,3,وحدة,...,8935770.0,40131.91,NaN,NaN,1,1,0,Q3-2025,2025,Flat
426,2-13-2018-1890,13,2,رهون,Mortgages,تسجيل رهن,Mortgage Registration,2018-03-01,2,مبنى,...,16000000.0,10330.04,NaN,NaN,1,1,0,Q1-2018,2018,Others
338,1-11-2004-356,11,1,مبايعات,Sales,بيع,Sell,2004-02-23,2,مبنى,...,21488440.0,4951.37,NaN,NaN,8,1,0,Q1-2004,2004,Others
490,1-11-2006-650,11,1,مبايعات,Sales,بيع,Sell,2006-04-25,4,فيلا,...,1300000.0,5541.82,NaN,NaN,5,1,0,Q2-2006,2006,Others
382,1-102-2024-29342,102,1,مبايعات,Sales,بيع - تسجيل مبدئى,Sell - Pre registration,2024-05-07,3,وحدة,...,3465000.0,26641.55,NaN,NaN,1,1,0,Q2-2024,2024,Flat
77,1-11-2003-1546,11,1,مبايعات,Sales,بيع,Sell,2003-10-20,4,فيلا,...,1350000.0,968.75,NaN,NaN,1,1,0,Q4-2003,2003,Others
304,3-9-2026-520,9,3,هبات,Gifts,هبه,Grant,2026-01-27,1,أرض,...,200000394.0,63627.93,NaN,NaN,1,1,0,Q1-2026,2026,Others
109,1-11-2005-1022,11,1,مبايعات,Sales,بيع,Sell,2005-04-18,2,مبنى,...,1400000.0,5935.22,NaN,NaN,1,1,0,Q2-2005,2005,Others
245,3-9-2002-78,9,3,هبات,Gifts,هبه,Grant,2002-07-01,2,مبنى,...,4500000.0,5734.97,NaN,NaN,1,1,0,Q3-2002,2002,Others
448,1-11-2006-530,11,1,مبايعات,Sales,بيع,Sell,2006-04-10,2,مبنى,...,8000000.0,62583.12,NaN,NaN,1,1,0,Q2-2006,2006,Others


In [11]:
df_transaction.to_excel(r"D:\Dubai\Dubai_Updated_data\Transection_data\DB1_Columns_Allignment\sample_transaction.xlsx", index=False)

# Column Name Update

In [6]:
import pandas as pd
import os

# ----------------------------------------------------------------------
# 1. Define the column mapping (Current_name -> Expected_name)
# ----------------------------------------------------------------------
mapping = [
    ("proj_id", "project_id"),
    ("project_number", "index"),
    ("project_name_en", "project_name"),
    ("village_mr", "village_name_marathi"),
    ("loc_id", "location_id"),
    ("area_name_en", "location_name"),
    ("area_name_ar", "location_name_ar"),
    ("igr_village", "village_name"),
    ("year", "year"),
    ("quarter", "quarter"),
    ("city_id", "city_id"),
    ("city", "city_name"),
    ("transaction_id", "document_number"),
    ("sro_code", "sub_registrar_office_code"),
    ("sro_name", "sub_registrar_office_name"),
    ("document_no", "document_number"),
    ("procedure_name_en", "transaction_type"),
    ("agreement_price", "agreement_price"),
    ("market_value", "guideline_value"),
    ("property_description", "property_description"),
    ("instance_date", "transaction_date"),
    ("floor_no", "floor_number"),
    ("unit_no", "unit_number"),
    ("procedure_area", "net_carpet_area_sq_m"),
    ("balcony_sqmt", "balcony_sq_m"),
    ("terrace_sqmt", "terrace_sq_m"),
    ("seller_name", "seller_name"),
    ("purchaser_name", "buyer_name"),
    ("property_category", "transaction_category"),
    ("internaldocumentnumber", "internal_document_number"),
    ("micrno", "micr_number"),
    ("bank_type", "bank_type"),
    ("party_code", "party_code"),
    ("dateofexecution", "date_of_agreement_execution"),
    ("stampdutypaid", "stamp_duty_paid"),
    ("registrationfees", "registration_fee"),
    (None, "project_latitude"),
    (None, "project_longitude"),
    (None, "location_latitude"),
    (None, "location_longitude"),
    ("property_sub_type_en", "property_type_raw"),
    ("rooms_en", "unit_configuration"),
    ("rooms_ar", "unit_configuration_ar"),
    ("buyer_pincode", "buyer_pincode"),
    ("trans_group_en", "transaction_category"),
    ("locality_of_buyer", "buyer_locality"),
    ("district", "buyer_district"),
    ("statename", "buyer_state"),
    (None, "is_llm_processed"),
    (None, "is_manual_processed"),
    ("building_name_en", "tower_name"),
    ("building_name_ar", "tower_name_ar"),
    ("gross_carpet_sqft", "gross_carpet_area_sq_ft"),
    ("rate_on_gca_sqft", "price_per_sq_ft_gross_carpet"),
    ("is_duplicate", "is_duplicate"),
    ("primary_sale_or_secondary_sale", "sale_type"),
    ("project_type", "project_type"),
    (None, "country_name"),
    (None, "state_name"),
    (None, "micro_market"),
    (None, "sub_locality"),
    (None, "pincode"),
    (None, "parking_count"),
    (None, "facing_direction"),
    (None, "view_type"),
    (None, "furnishing_status"),
    (None, "condition_status"),
    (None, "source_accessibility"),
    (None, "source_accessibility_way"),
    (None, "sourcing_cost"),
    (None, "sourcing_time"),
    (None, "data_type"),
    (None, "data_source"),
]

rename_dict = {curr: exp for curr, exp in mapping if curr is not None}
columns_to_create = [exp for curr, exp in mapping if curr is None]

# ----------------------------------------------------------------------
# 2. Read the input DataFrame
# ----------------------------------------------------------------------
input_path = r"D:\Dubai\Dubai_Updated_data\Transection_data\DB1_Columns_Allignment\sample_transaction.xlsx"
df = df_transaction.copy()

print(f"Original DataFrame shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")

# ----------------------------------------------------------------------
# 3. Rename columns
# ----------------------------------------------------------------------
df.rename(columns=rename_dict, inplace=True)

# ----------------------------------------------------------------------
# 4. Create missing columns
# ----------------------------------------------------------------------
for col in columns_to_create:
    if col not in df.columns:
        df[col] = None
        print(f"Created missing column: {col}")

# ----------------------------------------------------------------------
# 5. Save as CSV (recommended for large data)
# ----------------------------------------------------------------------
output_dir = os.path.dirname(input_path)
output_path = os.path.join(output_dir, "DB1.csv")

print("\nSaving full dataset into single CSV file...")

df.to_csv(
    output_path,
    index=False,
    encoding='utf-8-sig'   # keep this if opening in Excel
)

print(f"\n✅ CSV file created successfully: {output_path}")
print(f"Total rows saved: {len(df):,}")

Original DataFrame shape: (1661875, 50)
Rows: 1,661,875, Columns: 50
Created missing column: project_latitude
Created missing column: project_longitude
Created missing column: location_latitude
Created missing column: location_longitude
Created missing column: is_llm_processed
Created missing column: is_manual_processed
Created missing column: country_name
Created missing column: state_name
Created missing column: micro_market
Created missing column: sub_locality
Created missing column: pincode
Created missing column: parking_count
Created missing column: facing_direction
Created missing column: view_type
Created missing column: furnishing_status
Created missing column: condition_status
Created missing column: source_accessibility
Created missing column: source_accessibility_way
Created missing column: sourcing_cost
Created missing column: sourcing_time
Created missing column: data_type
Created missing column: data_source

Saving full dataset into single CSV file...

✅ CSV file created

# Column Order allignment

In [1]:
import pandas as pd

# Define the columns you want first
columns_first = [
    'index',
    'project_name',
    'project_name_ar',
    'quarter',
    'year',
    'location_name',
    'location_name_ar',
    'city_name',
    'property_type',
    'project_latitude',
    'project_longitude',
    'location_latitude',
    'location_longitude'
]

# Read your file
df = pd.read_csv(r"D:\Dubai\Dubai_Updated_data\Transection_data\DB1_Columns_Allignment\DB1.csv")

# Step 1: Add city_name column with constant value
df['city_name'] = "Dubai"
df['state_name'] = "Dubai"
df['country_name'] = "United Arab Emirates"

# Step 2: Get existing columns from your desired list
existing_first = [col for col in columns_first if col in df.columns]

# Step 3: Reorder columns
df = df[existing_first + [col for col in df.columns if col not in existing_first]]

# Save back to CSV
df.to_csv(r'D:\Dubai\Dubai_Updated_data\Transection_data\DB1_Columns_Allignment\DB1_Reord.csv', index=False)

print("✅ Columns reordered successfully!")
print(f"First 10 columns: {list(df.columns[:10])}")

✅ Columns reordered successfully!
First 10 columns: ['index', 'project_name', 'project_name_ar', 'quarter', 'year', 'location_name', 'location_name_ar', 'city_name', 'property_type', 'project_latitude']


In [2]:
df[['country_name','city_name','state_name']].head()

,country_name,city_name,state_name
0,United Arab Emirates,Dubai,Dubai
1,United Arab Emirates,Dubai,Dubai
2,United Arab Emirates,Dubai,Dubai
3,United Arab Emirates,Dubai,Dubai
4,United Arab Emirates,Dubai,Dubai


In [3]:
import pandas as pd

df_sample = pd.read_csv(r'D:\Dubai\Dubai_Updated_data\Transection_data\DB1_Columns_Allignment\DB1_Reord.csv', nrows=500)  

In [22]:
df_sample.head()

,index,project_name,project_name_ar,quarter,year,location_name,location_name_ar,city_name,property_type,project_latitude,...,facing_direction,view_type,furnishing_status,condition_status,source_accessibility,source_accessibility_way,sourcing_cost,sourcing_time,data_type,data_source
0,NaN,NaN,NaN,Q4-2006,2006.0,Mankhool,منخول,Dubai,Others,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,Q4-2019,2019.0,Mankhool,منخول,Dubai,Others,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,Q1-1999,1999.0,Mankhool,منخول,Dubai,Others,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,Q3-2001,2001.0,Oud Metha,عود ميثا,Dubai,Others,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,Q4-2020,2020.0,Al Bada,البدع,Dubai,Others,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
